# Neural Network V6

I compared 5, 15, and 30 training passes. Each pass goes through all the training examples once. I kept the network and data the same.

In [1]:
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
import torch
from torch import nn

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data_loader import load_task, split_train_val, standardize
from src.neural_net import (
    ModelConfig,
    evaluate,
    parameter_count,
    predict_probabilities,
    train_model,
)

epoch_counts = (5, 15, 30)
device = torch.device('cpu')


I started with the image data.

In [2]:
def run_epoch_sweep(task):
    X, y, _, _ = load_task(task, data_dir=ROOT / 'data')
    X_train, X_val, y_train, y_val = split_train_val(
        X, y, val_fraction=0.2, seed=42
    )
    X_train, y_train = X_train[:1000], y_train[:1000]
    X_val, y_val = X_val[:200], y_val[:200]
    X_train, X_val = standardize(X_train, X_val)

    output_size = 50 if task == 'image' else 3
    config = ModelConfig(
        hidden_sizes=(64,),
        optimizer='adam',
        learning_rate=0.001,
        batch_size=32,
    )
    rows = []
    predictions_by_epochs = {}

    for epochs in epoch_counts:
        started = time.perf_counter()
        result = train_model(
            X_train,
            y_train,
            output_size,
            config,
            epochs,
            seed=42,
            device=device,
            validation_features=X_val,
            validation_labels=y_val,
            early_stopping=False,
        )
        training_time = time.perf_counter() - started

        _, training_accuracy = evaluate(
            result.model,
            X_train,
            y_train,
            config.batch_size,
            device,
            nn.CrossEntropyLoss(),
        )
        prediction_started = time.perf_counter()
        probabilities = predict_probabilities(
            result.model, X_val, config.batch_size, device
        )
        prediction_time = time.perf_counter() - prediction_started
        predictions = probabilities.argmax(axis=1)
        validation_accuracy = float(np.mean(predictions == y_val))
        if not np.isfinite(validation_accuracy):
            raise RuntimeError(f'{task} validation accuracy is not finite')

        rows.append({
            'version': 6,
            'task': task,
            'epochs': epochs,
            'best_epoch': result.best_epoch,
            'config': (
                'hidden_sizes=[64]; activation=relu; optimizer=adam; '
                'learning_rate=0.001; batch_size=32; normalized=true; '
                'training_samples=1000; validation_samples=200'
            ),
            'parameters': parameter_count(result.model),
            'training_time_seconds': training_time,
            'prediction_time_seconds': prediction_time,
            'best_val_accuracy': validation_accuracy,
            'final_train_accuracy': float(training_accuracy),
            'train_val_gap': float(training_accuracy - validation_accuracy),
        })
        predictions_by_epochs[epochs] = predictions

    best = max(
        rows,
        key=lambda row: (row['best_val_accuracy'], -row['epochs']),
    )
    return {
        'rows': rows,
        'actual_labels': y_val,
        'predictions': predictions_by_epochs[best['epochs']],
    }


In [3]:
image_results = run_epoch_sweep('image')
print(pd.DataFrame(image_results['rows']).to_string(index=False))


 version  task  epochs  best_epoch                                                                                                                                                 config  parameters  training_time_seconds  prediction_time_seconds  best_val_accuracy  final_train_accuracy  train_val_gap
       6 image       5           5 hidden_sizes=[64]; activation=relu; optimizer=adam; learning_rate=0.001; batch_size=32; normalized=true; training_samples=1000; validation_samples=200       35314               3.233726                 0.001934              0.095                 0.806          0.711
       6 image      15          15 hidden_sizes=[64]; activation=relu; optimizer=adam; learning_rate=0.001; batch_size=32; normalized=true; training_samples=1000; validation_samples=200       35314               0.992595                 0.002043              0.135                 1.000          0.865
       6 image      30          26 hidden_sizes=[64]; activation=relu; optimizer=adam; learnin

Then I repeated the test with the sentiment data.

In [4]:
sentiment_results = run_epoch_sweep('sentiment')
print(pd.DataFrame(sentiment_results['rows']).to_string(index=False))


 version      task  epochs  best_epoch                                                                                                                                                 config  parameters  training_time_seconds  prediction_time_seconds  best_val_accuracy  final_train_accuracy  train_val_gap
       6 sentiment       5           3 hidden_sizes=[64]; activation=relu; optimizer=adam; learning_rate=0.001; batch_size=32; normalized=true; training_samples=1000; validation_samples=200       49411               0.442163                 0.002287               0.61                 0.882          0.272
       6 sentiment      15           3 hidden_sizes=[64]; activation=relu; optimizer=adam; learning_rate=0.001; batch_size=32; normalized=true; training_samples=1000; validation_samples=200       49411               1.007699                 0.003120               0.61                 0.882          0.272
       6 sentiment      30           3 hidden_sizes=[64]; activation=relu; optimiz

I saved the results and predictions from the best run.

In [5]:
for task, task_results in (
    ('image', image_results),
    ('sentiment', sentiment_results),
):
    summary = pd.DataFrame(task_results['rows'])
    summary.to_csv(
        ROOT / 'results' / task / 'nn_v6_summary.csv',
        index=False,
    )
    pd.DataFrame({
        'validation_index': np.arange(len(task_results['actual_labels'])),
        'actual_label': task_results['actual_labels'],
        'predicted_label': task_results['predictions'],
    }).to_csv(
        ROOT / 'results' / task / 'nn_v6_validation.csv',
        index=False,
    )
    best = max(
        task_results['rows'],
        key=lambda row: (row['best_val_accuracy'], -row['epochs']),
    )
    print(f"{task}: best result used {best['epochs']} training passes")


image: best result used 30 training passes
sentiment: best result used 5 training passes
